# tickdb demo

End-to-end walkthrough of the HTTP API:
1. Start `tickdb-server`
2. Ingest synthetic trades via `POST /ingest`
3. Query OHLCV bars, VWAP, and an as-of join
4. Load Arrow IPC responses directly into pandas

**Prerequisites**
```bash
cargo run -p server --release -- --data-dir /tmp/tickdb-demo --port 8080 --fsync never &
pip install requests pandas pyarrow
```

In [ ]:
import requests
import pyarrow as pa
import pyarrow.ipc as ipc
import pandas as pd
import io

BASE = "http://127.0.0.1:8080"

def arrow_get(path):
    resp = requests.get(f"{BASE}{path}")
    resp.raise_for_status()
    reader = ipc.open_stream(io.BytesIO(resp.content))
    return reader.read_pandas()

## 1. Ingest synthetic trades

Generate a small synthetic tick stream for `BTCUSDT` and `ETHUSDT` with a
tight random walk in price, and POST it to `/ingest`.

In [ ]:
import random

def gen_trades(n, base_ts, base_price_mantissa, exponent=-2, seed=0):
    rng = random.Random(seed)
    ts = base_ts
    price = base_price_mantissa
    trades = []
    for i in range(n):
        ts += rng.randint(950_000_000, 1_050_000_000)  # ~1s apart, in ns
        price += rng.choice([-1, 0, 0, 1])
        trades.append({
            "ts": ts,
            "price_mantissa": price,
            "price_exponent": exponent,
            "size": rng.randint(1, 100),
            "side": rng.choice([0, 1]),
            "trade_id": i,
        })
    return trades

BASE_TS = 1_705_276_800_000_000_000  # 2024-01-15 00:00:00 UTC

btc_trades = gen_trades(2000, BASE_TS, 4_325_100, exponent=-2, seed=1)
eth_trades = gen_trades(2000, BASE_TS, 2_250_000, exponent=-2, seed=2)

r1 = requests.post(f"{BASE}/ingest", json={"symbol": "BTCUSDT", "trades": btc_trades})
r2 = requests.post(f"{BASE}/ingest", json={"symbol": "ETHUSDT", "trades": eth_trades})
print(r1.json(), r2.json())

## 2. OHLCV bars

`GET /query/bars` returns an Arrow IPC stream — `pyarrow` loads it directly
into a pandas DataFrame with no custom parsing.

In [ ]:
range_end = BASE_TS + 2000 * 1_100_000_000
interval_ns = 60_000_000_000  # 1-minute bars

bars = arrow_get(f"/query/bars?symbol=BTCUSDT&start={BASE_TS}&end={range_end}&interval_ns={interval_ns}")
bars["datetime"] = pd.to_datetime(bars["ts"], unit="ns")
bars.head()

In [ ]:
# Quick OHLC plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bars["datetime"], bars["close"], label="close")
ax.fill_between(bars["datetime"], bars["low"], bars["high"], alpha=0.2, label="high/low range")
ax.set_title("BTCUSDT 1-minute bars")
ax.legend()
plt.show()

## 3. VWAP

`GET /query/vwap` returns plain JSON (a single scalar doesn't need Arrow).

In [ ]:
vwap = requests.get(f"{BASE}/query/vwap?symbol=BTCUSDT&start={BASE_TS}&end={range_end}").json()
print(f"BTCUSDT VWAP over the full range: {vwap['vwap']:.2f}")

## 4. As-of join

Attach the most recent ETHUSDT trade (within 2 seconds) to every BTCUSDT trade.
Useful for cross-asset lead/lag analysis.

In [ ]:
tolerance_ns = 2_000_000_000  # 2 seconds
joined = arrow_get(
    f"/query/asof?left=BTCUSDT&right=ETHUSDT&start={BASE_TS}&end={range_end}&tolerance_ns={tolerance_ns}"
)

match_rate = joined["right_ts"].notna().mean()
print(f"match rate within {tolerance_ns/1e9:.0f}s tolerance: {match_rate:.1%}")
joined.head()

## 5. Raw trades + stats

In [ ]:
trades_df = arrow_get(f"/query/trades?symbol=BTCUSDT&start={BASE_TS}&end={range_end}")
print(f"{len(trades_df)} trades loaded")
trades_df.head()

In [ ]:
# Force a flush so the data is durably on disk, then check stats.
requests.post(f"{BASE}/admin/flush")
requests.get(f"{BASE}/stats").json()